# 01. Exploratory Data Analysis: MEPS 2024 HC-256
## Research Project: Uncertainty-Aware Predictive Modeling Framework for Healthcare Cost Prediction

**Author**: Senior Research Data Scientist  
**Dataset**: Medical Expenditure Panel Survey (MEPS), 2024 Full Year Consolidated Data (HC-256), Agency for Healthcare Research and Quality (AHRQ)  
**Objective**: Characterize the statistical properties, zero-inflation, extreme right-skewness, and subgroup disparities in annual out-of-pocket healthcare expenditures (`TOTSLF24`).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# Ensure workspace src is importable
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root))

from src.visualization import plot_target_distribution, plot_expenditure_by_subgroup
from src.data_preprocessing import ALL_PREDICTORS, NUMERIC_FEATURES, CATEGORICAL_FEATURES, FEATURE_VALUE_LABELS

print("Environment and modules initialized.")


In [ ]:
data_path = repo_root / "data" / "processed" / "meps_2024_cleaned.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded cleaned analytic cohort with {df.shape[0]:,} individuals and {df.shape[1]} columns.")
df.head()


### 1. Target Variable Analysis: Out-of-Pocket Healthcare Costs (`TOTSLF24`)
We examine the distribution, zero mass, percentiles, skewness, and kurtosis.


In [ ]:
y = df["TOTSLF24"]
y_log = np.log1p(y)

print("=== Raw Out-of-Pocket Expenditure ($ USD) ===")
print(f"Mean:              ${y.mean():,.2f}")
print(f"Standard Dev:      ${y.std():,.2f}")
print(f"Median (50th):     ${y.median():,.2f}")
print(f"25th Percentile:   ${np.percentile(y, 25):,.2f}")
print(f"75th Percentile:   ${np.percentile(y, 75):,.2f}")
print(f"90th Percentile:   ${np.percentile(y, 90):,.2f}")
print(f"99th Percentile:   ${np.percentile(y, 99):,.2f}")
print(f"Max Expenditure:   ${y.max():,.2f}")
print(f"Zero Spenders:     {(y == 0).sum():,} ({(y == 0).mean():.2%})")
print(f"Skewness:          {stats.skew(y):.2f}")
print(f"Excess Kurtosis:   {stats.kurtosis(y):.2f}")

print("
=== Log-Transformed [log(1 + y)] Scale ===")
print(f"Mean:              {y_log.mean():.3f}")
print(f"Std:               {y_log.std():.3f}")
print(f"Median:            {y_log.median():.3f}")
print(f"Skewness:          {stats.skew(y_log):.2f}")


### 2. Subgroup Disparities: Insurance, Age, and Health Status
Healthcare expenditures vary markedly across insurance coverage types, age groups, and chronic disease comorbidities.


In [ ]:
# Expenditure by Insurance Category
ins_summary = df.groupby("INSCOV24")["TOTSLF24"].agg(
    Count="count",
    Mean_Spend="mean",
    Median_Spend="median",
    Zero_Rate=lambda x: (x == 0).mean() * 100
).reset_index()
ins_summary["Category"] = ins_summary["INSCOV24"].map(FEATURE_VALUE_LABELS["INSCOV24"])
ins_summary


In [ ]:
# Expenditure by Perceived Health Status (1=Excellent to 5=Poor)
hlth_summary = df.groupby("RTHLTH31")["TOTSLF24"].agg(
    Count="count",
    Mean_Spend="mean",
    Median_Spend="median",
    Zero_Rate=lambda x: (x == 0).mean() * 100
).reset_index()
hlth_summary["Health_Status"] = hlth_summary["RTHLTH31"].map(FEATURE_VALUE_LABELS["RTHLTH31"])
hlth_summary


### Summary Findings for Conformal Modeling:
1. **Zero Inflation & Right Skew**: 26.11% of patients incur $0 out-of-pocket costs, while the top 1% incur upwards of $17,000 to $317,942.
2. **Heteroscedasticity**: Conditional variance scales with age, income, and chronic burden. Standard constant-width prediction intervals fail to capture this variance structure.
3. **Normalized Conformal Prediction**: Implementing an adaptive residual scale model $s(x)$ is necessary to achieve valid, heteroscedastic prediction intervals.
